In [4]:
import gymnasium as gym
from gymnasium.spaces import Box
import numpy as np
from typing import Optional, Any

In [11]:
class ContinuousStateContinuousAction(gym.Env):
    """Пример среды: непрерывные состояния, п=непрерывные действия"""
    
    metadata = {"renedr_modes": ["human"], "render_fps": 4}
    
    def __init__(self, 
                 *, 
                 render_mode: Optional[str] = None, 
                 seed: Optional[int] = None,
                 max_steps: int = 10
                 ) -> None:
        
        super().__init__()
        
        # Непрерывное пространство состояний
        self.observation_space = Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32,
        )
        
        # Непрерывное пространство действий
        self.action_space = Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32
        )
        
        # Инициализация
        self.state = np.array([0.0], dtype=np.float32)
        self.step_count = 0
        
        # Параметры визуализации
        self.render_mode = render_mode
        
        # Инициализация случайности
        self.np_random, _ = gym.utils.seeding.np_random(seed)

        self.max_steps = max_steps
        
    # -----------------------
    # Основные методы Gym API
    # -----------------------

    def reset(
        self,
        *,
        seed: Optional[int] = None,
        options: Optional[dict[str, Any]] = None,
    ):
        super().reset(seed=seed)
        self.state = self.np_random.uniform(low=-0.5, high=0.5, size=(1,)).astype(np.float32)
        self.step_count = 0
        info = {}

        if self.render_mode == "human":
            print(f"[reset] начальное состояние: {self.state[0]:.3f}")

        return self.state, info

    def step(self, action: np.float32):
        # Проверка допустимости действия
        assert self.action_space.contains(action), f"Неверное действие: {action}"

        # Динамика
        noise = self.np_random.normal(0, 0.01)
        self.state = np.clip(self.state + action + noise, -1.0, 1.0)

        # Вознаграждение — чем ближе к 0, тем лучше
        reward = -abs(self.state[0])

        self.step_count += 1

        # 👇 Termination по числу шагов
        terminated = self.step_count >= self.max_steps
        truncated = False   # Можно добавить ограничение по времени
        info = {}

        if self.render_mode == "human":
            print(
                f"[step {self.step_count:02d}] "
                f"action={action[0]:+0.3f} "
                f"state={self.state[0]:+0.3f} "
                f"reward={reward:+0.3f}"
            )

        return self.state, reward, terminated, truncated, info

    def render(self):
        print(f"Текущее состояние: {self.state[0]:+.3f}")

    def close(self):
        pass

In [12]:
if __name__ == "__main__":
    env = ContinuousStateContinuousAction(render_mode="human", max_steps=10)
    obs, info = env.reset()

    done = False
    while not done:
        action = env.action_space.sample()  # случайное непрерывное действие
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

    env.close()

[reset] начальное состояние: -0.307
[step 01] action=-0.593 state=-0.887 reward=-0.887
[step 02] action=+0.030 state=-0.863 reward=-0.863
[step 03] action=-0.037 state=-0.890 reward=-0.890
[step 04] action=-0.214 state=-1.000 reward=-1.000
[step 05] action=-0.947 state=-1.000 reward=-1.000
[step 06] action=+0.123 state=-0.871 reward=-0.871
[step 07] action=+0.019 state=-0.865 reward=-0.865
[step 08] action=+0.650 state=-0.221 reward=-0.221
[step 09] action=-0.552 state=-0.754 reward=-0.754
[step 10] action=+0.203 state=-0.557 reward=-0.557
